# Viabilidade — modelo a nível de município (retomando essa ideia depois do achado da escola)

Confirmei no `04_eda_escola_completo.ipynb` que a `escola_completo` (455 colunas) não tem nenhum indicador de resultado/desempenho educacional nativo — não dá pra montar uma variável-alvo por escola sem depender do JOIN com `alunos`, que por sua vez está confirmado inviável (`03_eda_escola.ipynb`, Seção 4.3 — dois sistemas de `id_escola` incompatíveis, sem crosswalk).

A alternativa que levantei pra discutir: retomar o **município** como unidade de predição. Diferente de `id_escola`, o `id_municipio` já funciona perfeitamente como chave de JOIN entre `alunos` e as demais tabelas — é o mesmo `id_municipio` que uso desde o Dia 1 pra trazer infraestrutura, metas e evolução temporal pro modelo por aluno. E com a `escola_completo` recém-disponível, dá pra agregar isso por município e ganhar bem mais features do que as 6 `pct_escolas_*` que já uso hoje (vindas da Gold `indicador_x_infraestrutura_escolar`).

A preocupação que também levantei: são só ~5.570 municípios no Brasil — bem menos que os quase 3,9 milhões de alunos ou as 215 mil escolas. Se eu adicionar muitas features (a escola agregada, mais tudo que já uso hoje), corro o risco de ter mais colunas do que dado suficiente pra treinar sem overfitar.

**Este notebook não decide a arquitetura — só levanta os números reais que faltam pra essa decisão:** quantos municípios têm alvo e features completos, e quantas features dá pra montar de forma razoável. Mesma disciplina de sempre: checar com dado real antes de decidir, não assumir.

In [ ]:
import sys
sys.path.append("../..")

import boto3
import pandas as pd
import numpy as np
from pathlib import Path

from src.preprocessing.load_data import (
    BUCKET,
    _ler_parquet_do_prefixo,
    ler_alunos,
    ler_infraestrutura_municipio,
    ler_evolucao_temporal_municipio,
    ler_metas_municipio,
    ler_municipio_completo,
)

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 100)

## 1. Carga dos dados

Reaproveito as funções que já existem em `load_data.py` (as mesmas que uso desde o Dia 1/Dia 2 pra montar a base por aluno) e a `escola_completo` que já cachei localmente no `04_eda_escola_completo.ipynb`.

In [ ]:
alunos = ler_alunos()

CACHE_ESCOLA_COMPLETO = Path("../..") / "data" / "processed" / "escola_completo.parquet"
if CACHE_ESCOLA_COMPLETO.exists():
    print(f"Lendo escola_completo do cache local: {CACHE_ESCOLA_COMPLETO}")
    escola_completo = pd.read_parquet(CACHE_ESCOLA_COMPLETO)
else:
    PREFIXO_BRONZE_ESCOLA_COMPLETO = "bronze/br_inep_censo_escolar/escola_completo/"
    print(f"Cache não encontrado, lendo do S3: s3://{BUCKET}/{PREFIXO_BRONZE_ESCOLA_COMPLETO}")
    escola_completo = _ler_parquet_do_prefixo(BUCKET, PREFIXO_BRONZE_ESCOLA_COMPLETO)
    CACHE_ESCOLA_COMPLETO.parent.mkdir(parents=True, exist_ok=True)
    escola_completo.to_parquet(CACHE_ESCOLA_COMPLETO, index=False)
    print(f"Cache salvo em: {CACHE_ESCOLA_COMPLETO}")

print(f"\nalunos: {alunos.shape[0]:,} linhas, {alunos.shape[1]} colunas")
print(f"escola_completo: {escola_completo.shape[0]:,} linhas, {escola_completo.shape[1]} colunas")

## 2. Quantos municípios existem de verdade nos dados, e quantos alunos cada um tem

Antes de pensar em features, preciso saber o tamanho real do problema: quantos municípios têm alunos avaliados, e qual a distribuição de alunos por município. Isso importa porque um município com poucos alunos avaliados tem um alvo (% de alfabetizados) estatisticamente instável — por exemplo, 2 alunos avaliados e 1 alfabetizado dá 50%, o que não quer dizer muita coisa sobre o município real.

In [ ]:
alunos_por_municipio = alunos.groupby("id_municipio").size().rename("qtd_alunos_avaliados")

print(f"Total de municípios distintos em 'alunos': {alunos_por_municipio.shape[0]:,}")
print("Total de municípios no Brasil, referência IBGE: 5.570")
print()
print("Distribuição de alunos avaliados por município:")
print(alunos_por_municipio.describe())

print("\nMunicípios com menos de 10 alunos avaliados:", (alunos_por_municipio < 10).sum())
print("Municípios com menos de 30 alunos avaliados:", (alunos_por_municipio < 30).sum())
print("Municípios com pelo menos 100 alunos avaliados:", (alunos_por_municipio >= 100).sum())

## 3. Alvo candidato: % de alunos alfabetizados por município

Mesmo cálculo que já usei na exploração de score de município (`02_exploracao_score_municipio.ipynb`) — direto da `alunos`, sem depender da tabela Gold pronta (que já vem filtrada em `rede='5'`, e eu quero ver o município como um todo primeiro, sem esse filtro embutido).

In [ ]:
alunos["alfabetizado"] = alunos["alfabetizado"].astype(int)

alvo_municipio = (
    alunos.groupby(["id_municipio", "ano"])["alfabetizado"]
    .mean()
    .mul(100)
    .rename("percentual_alfabetizados")
    .reset_index()
)

print(f"Linhas (município, ano): {len(alvo_municipio):,}")
print(f"Municípios distintos: {alvo_municipio['id_municipio'].nunique():,}")
print("\nDistribuição do percentual de alfabetizados por (município, ano):")
print(alvo_municipio["percentual_alfabetizados"].describe())

## 4. O que já existe pronto como feature de município (sem tocar em `escola_completo` ainda)

Já uso essas quatro fontes desde o Dia 1/Dia 2 pra enriquecer o modelo por aluno via JOIN em `id_municipio` (ou `id_municipio, ano`). Elas já removem colunas de leakage por agregação (documentado nos comentários do `load_data.py`) — só preciso ver quantas colunas cada uma realmente traz, pra somar no total de features candidatas.

In [ ]:
infra_municipio = ler_infraestrutura_municipio()
evolucao_municipio = ler_evolucao_temporal_municipio()
metas_municipio = ler_metas_municipio()
municipio_completo = ler_municipio_completo()

print("\nResumo de colunas por fonte (excluindo as chaves id_municipio/ano):")
for nome, df in [
    ("infraestrutura (Gold)", infra_municipio),
    ("evolução temporal (Gold)", evolucao_municipio),
    ("metas (Silver)", metas_municipio),
    ("município completo (Silver)", municipio_completo),
]:
    colunas_feature = [c for c in df.columns if c not in ("id_municipio", "ano")]
    print(f"- {nome}: {len(colunas_feature)} colunas -> {colunas_feature}")

## 5. Agregando `escola_completo` por município — o ganho novo

Isso é uma primeira passada exploratória, não a engenharia de feature final (a pendência 13 já prevê tratar as 455 colunas com mais cuidado, transformando contagem bruta em proporção/razão). A ideia aqui é só medir o tamanho do ganho: quantas features a mais dá pra montar agregando a escola por município, comparado às 6 `pct_escolas_*` que já uso hoje.

Trato os dois tipos de coluna de forma diferente:
- Colunas binárias (`0`/`1`, "tem ou não tem" infraestrutura) viram **% de escolas do município que têm aquilo** (`.mean()`).
- Colunas de contagem (`quantidade_*`) viram **soma no município** (dá pra eu transformar em razão depois, ex. aluno/professor — fora do escopo deste notebook).

In [ ]:
escola_completo["id_municipio"] = escola_completo["id_municipio"].astype(str)

colunas_identificacao = [
    "ano", "sigla_uf", "id_municipio", "id_escola", "rede",
    "tipo_localizacao", "tipo_situacao_funcionamento",
]

colunas_binarias = [
    c for c in escola_completo.columns
    if c not in colunas_identificacao
    and escola_completo[c].dropna().isin([0, 1]).all()
    and escola_completo[c].nunique(dropna=True) <= 2
]

colunas_quantidade = [
    c for c in escola_completo.columns
    if c.startswith("quantidade_") and c not in colunas_binarias
]

print(f"Colunas binárias identificadas (viram % de escolas no município): {len(colunas_binarias)}")
print(f"Colunas de contagem identificadas (viram soma no município): {len(colunas_quantidade)}")

agregado_binario = escola_completo.groupby("id_municipio")[colunas_binarias].mean().add_prefix("pct_escolas_")
agregado_quantidade = escola_completo.groupby("id_municipio")[colunas_quantidade].sum().add_prefix("soma_")
quantidade_escolas_municipio = escola_completo.groupby("id_municipio").size().rename("quantidade_escolas_no_municipio")

escola_agregada_municipio = pd.concat(
    [quantidade_escolas_municipio, agregado_binario, agregado_quantidade], axis=1
).reset_index()

print(f"\nescola_agregada_municipio: {escola_agregada_municipio.shape[0]:,} municípios, {escola_agregada_municipio.shape[1] - 1} features novas")

## 6. Checando a interseção `id_municipio` entre `alunos` e `escola_completo`

Depois da lição do `id_escola` (que parecia um código válido mas eram dois sistemas diferentes), não vou assumir que `id_municipio` bate só porque teoricamente os dois usam o código IBGE — confiro com número real dos dois lados antes de confiar nessa chave pra uma decisão de arquitetura.

In [ ]:
municipios_alunos = set(alunos["id_municipio"].astype(str).unique())
municipios_escola = set(escola_completo["id_municipio"].astype(str).unique())

interseccao = municipios_alunos & municipios_escola

print(f"Municípios distintos em 'alunos': {len(municipios_alunos):,}")
print(f"Municípios distintos em 'escola_completo': {len(municipios_escola):,}")
print(f"Interseção: {len(interseccao):,}")
print(f"% dos municípios de 'alunos' que têm correspondência em 'escola_completo': {len(interseccao) / len(municipios_alunos) * 100:.2f}%")

print("\nAmostra de id_municipio em 'alunos':", sorted(municipios_alunos)[:5])
print("Amostra de id_municipio em 'escola_completo':", sorted(municipios_escola)[:5])

## 7. Consolidando: quantas features, quantos municípios, qual razão observação/feature

Junto tudo (alvo + as quatro fontes que já existiam + a agregação nova da escola) num único `DataFrame` por município, só pra contar de verdade — sem ainda decidir quais features ficam, como tratar nulos, ou resolver multicolinearidade (isso é trabalho de feature engineering de verdade, fora do escopo deste notebook de viabilidade).

In [ ]:
base_municipio = alvo_municipio.merge(infra_municipio, on=["id_municipio", "ano"], how="left")
base_municipio = base_municipio.merge(evolucao_municipio, on=["id_municipio", "ano"], how="left")

chaves_metas = [c for c in ["id_municipio", "ano"] if c in metas_municipio.columns]
base_municipio = base_municipio.merge(metas_municipio, on=chaves_metas, how="left")

chaves_municipio_completo = [c for c in ["id_municipio", "ano"] if c in municipio_completo.columns]
base_municipio = base_municipio.merge(municipio_completo, on=chaves_municipio_completo, how="left")

base_municipio = base_municipio.merge(escola_agregada_municipio, on="id_municipio", how="left")

colunas_feature = [c for c in base_municipio.columns if c not in ("id_municipio", "ano", "percentual_alfabetizados")]

print(f"Linhas (município, ano) na base consolidada: {len(base_municipio):,}")
print(f"Municípios distintos: {base_municipio['id_municipio'].nunique():,}")
print(f"Total de colunas candidatas a feature: {len(colunas_feature)}")

linhas_com_alvo = base_municipio.dropna(subset=["percentual_alfabetizados"])
print(f"\nLinhas (município, ano) com alvo definido: {len(linhas_com_alvo):,}")
print(f"Razão observações / features candidatas: {len(linhas_com_alvo) / len(colunas_feature):.1f} : 1")

## 8. O que essa razão significa (referência pra discussão, não uma decisão)

Não existe uma regra fixa e universal, mas algumas referências úteis pra essa conversa:

- Modelos baseados em árvore (Random Forest, Gradient Boosting) toleram razões observação:feature mais apertadas que modelos lineares, principalmente com regularização e validação cruzada — mas abaixo de aproximadamente 10:1 o risco de overfitting fica alto mesmo assim.
- Modelos lineares/logísticos com muitas variáveis correlacionadas (bem provável aqui, já que várias colunas de infraestrutura tendem a andar juntas) geralmente precisam de mais margem, ou de regularização (L1/L2) e/ou redução de dimensionalidade (ex. PCA, seleção de features) pra não overfitar.
- Se a razão ficar apertada, um caminho possível é não usar as centenas de colunas agregadas cruas, e sim um conjunto bem mais enxuto e selecionado (a pendência 13 já previa esse tratamento), ou reduzir dimensionalidade antes de modelar.

Fica aqui como referência pra essa decisão, não como uma recomendação fechada de que devo seguir com um número específico de features.

## 9. Conclusão — dá pra treinar um modelo a nível de município?

Não cheguei a rodar esse notebook até o fim com a base real: antes de terminar de levantar esses números, o `04_eda_escola_completo.ipynb` já tinha me mostrado que a `escola_completo` não tem indicador de resultado nativo, e a busca por uma fonte externa de desempenho por escola me levou direto ao `br_inep_ideb`, que virou a base do meu alvo definitivo em `06_eda_ideb_escola.ipynb`. Com um indicador de desempenho já disponível diretamente no grão de escola, a alternativa de "subir" pro grão de município — que era, no fundo, uma forma de contornar a falta de alvo no grão de escola trocando granularidade por viabilidade de dado — deixou de ser necessária.

Não descarto essas ideias por serem ruins: a checagem que fica pendente aqui (tamanho da amostra por município, razão observação/feature, interseção com `escola_completo`) é um exercício de bom senso que continuaria válido se o projeto precisasse revisitar essa direção no futuro. Mas com o IDEB resolvendo o problema de alvo diretamente no grão de escola — que também é a granularidade mais acionável, porque dá pra transformar o resultado do modelo em plano de ação concreto por escola, algo bem mais difícil de fazer a nível de município — essa investigação específica fica registrada aqui como uma alternativa considerada e não seguida, não como uma pendência em aberto.